In [93]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import *
import joblib

In [79]:
lead_scoring=pd.read_csv('lead_scoring_final.csv')

In [80]:
lead_scoring.head()

,Customer ID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,Segment,AvgOrderValue,PurchaseIntensity,ProductDiversity,Customer_active_days,email_click_rate,website_visits,time_on_site,discount_response,LeadScore,LeadScore_Log,LeadScore_Final
0,12346.0,325,12,77556.46,2,5,5,255,Loyal Customer,6463.038333,0.036810,27,400,1.0,33,330,0,254.368098,5.542706,30.664569
1,12347.0,1,8,5633.32,5,4,5,545,Champion,704.165000,4.000000,126,402,0.9,24,288,1,520.000000,6.255750,44.405869
2,12348.0,74,5,2019.40,3,4,4,344,Loyal Customer,403.880000,0.066667,25,362,0.8,12,84,0,231.666667,5.449607,28.870424
3,12349.0,18,4,4428.69,5,3,5,535,Champion,1107.172500,0.210526,138,570,0.8,10,80,1,489.105263,6.194620,43.227816
4,12350.0,309,1,334.40,2,1,2,212,Potential Customer,334.400000,0.003226,17,0,0.3,11,99,0,115.032258,4.753868,15.462621


In [81]:
lead_scoring.shape

(5878, 20)

In [82]:
ml_data = lead_scoring.copy()

In [83]:
X =ml_data[[
    'Recency',
    'Frequency',
    'Monetary',
    'ProductDiversity',
    'Customer_active_days',
    'email_click_rate',
    'website_visits',
    'time_on_site',
    'discount_response'
]]

In [84]:
Y=ml_data['LeadScore_Final']

In [85]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

In [86]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

### Logisitic Regression

In [95]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 🔥 TRAIN
lr_model = LinearRegression()
lr_model.fit(X_train, Y_train)

# 🔥 PREDICT
y_pred_lr = lr_model.predict(X_test)

# ================================
# 📊 METRICS
# ================================
mae = mean_absolute_error(Y_test, y_pred_lr)
mse = mean_squared_error(Y_test, y_pred_lr)
rmse = np.sqrt(mse)
r2 = r2_score(Y_test, y_pred_lr)

lr_results = {
    "Model": "Linear Regression",
    "MAE": mae,
    "MSE": mse,
    "RMSE": rmse,
    "R2 Score": r2
}

print(lr_results)

pd.DataFrame([lr_results]).to_csv("lr_regression_metrics.csv", index=False)

# ================================
# 📉 ACTUAL VS PREDICTED
# ================================
plt.figure()
plt.scatter(Y_test, y_pred_lr)
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("Actual vs Predicted - Linear Regression")
plt.savefig("actual_vs_pred_lr.png")
plt.close()

# ================================
# 📊 RESIDUAL PLOT
# ================================
residuals = Y_test - y_pred_lr

plt.figure()
plt.scatter(y_pred_lr, residuals)
plt.xlabel("Predicted")
plt.ylabel("Residuals")
plt.title("Residual Plot - Linear Regression")
plt.savefig("residual_plot_lr.png")
plt.close()

# ================================
# 💾 SAVE MODEL
# ================================
joblib.dump(lr_model, "linear_regression_model.pkl")

{'Model': 'Linear Regression', 'MAE': 2.818671597480719, 'MSE': 14.6966284357465, 'RMSE': 3.833618191180037, 'R2 Score': 0.9250243359476306}


['linear_regression_model.pkl']

In [99]:
importance = lr_model.coef_

importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": importance
}).sort_values(by="Importance", ascending=False)

print(importance_df)

                Feature  Importance
5      email_click_rate    8.165633
3      ProductDiversity    6.704659
8     discount_response    0.596823
2              Monetary    0.419936
6        website_visits    0.048726
7          time_on_site   -0.110813
4  Customer_active_days   -0.210754
1             Frequency   -1.936302
0               Recency   -2.231632


In [100]:
importance_df.to_csv("feature_importance_lr.csv", index=False)

In [96]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 🔥 TRAIN
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=5,
    random_state=42
)

rf_model.fit(X_train, Y_train)

# 🔥 PREDICT
y_pred_rf = rf_model.predict(X_test)

# ================================
# 📊 METRICS
# ================================
mae = mean_absolute_error(Y_test, y_pred_rf)
mse = mean_squared_error(Y_test, y_pred_rf)
rmse = np.sqrt(mse)
r2 = r2_score(Y_test, y_pred_rf)

rf_results = {
    "Model": "Random Forest Regressor",
    "MAE": mae,
    "MSE": mse,
    "RMSE": rmse,
    "R2 Score": r2
}

print(rf_results)

pd.DataFrame([rf_results]).to_csv("rf_regression_metrics.csv", index=False)

# ================================
# 📉 ACTUAL VS PREDICTED
# ================================
plt.figure()
plt.scatter(Y_test, y_pred_rf)
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("Actual vs Predicted - Random Forest")
plt.savefig("actual_vs_pred_rf.png")
plt.close()

# ================================
# 📊 RESIDUAL PLOT
# ================================
residuals = Y_test - y_pred_rf

plt.figure()
plt.scatter(y_pred_rf, residuals)
plt.xlabel("Predicted")
plt.ylabel("Residuals")
plt.title("Residual Plot - Random Forest")
plt.savefig("residual_plot_rf.png")
plt.close()

# ================================
# 📈 FEATURE IMPORTANCE
# ================================
feature_importance = rf_model.feature_importances_

importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": feature_importance
}).sort_values(by="Importance", ascending=False)

importance_df.to_csv("feature_importance_rf.csv", index=False)

plt.figure()
plt.barh(importance_df["Feature"], importance_df["Importance"])
plt.title("Feature Importance - Random Forest")
plt.gca().invert_yaxis()
plt.savefig("feature_importance_rf.png")
plt.close()

# ================================
# 💾 SAVE MODEL
# ================================
joblib.dump(rf_model, "random_forest_model.pkl")

{'Model': 'Random Forest Regressor', 'MAE': 1.4430453125229932, 'MSE': 3.729257248411216, 'RMSE': 1.9311284909117818, 'R2 Score': 0.980974987573227}


['random_forest_model.pkl']

In [97]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# 🔥 TRAIN MODEL
gb_model = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

gb_model.fit(X_train, Y_train)

# 🔥 PREDICT
y_pred_gb = gb_model.predict(X_test)

# ================================
# 📊 REGRESSION METRICS
# ================================

mae = mean_absolute_error(Y_test, y_pred_gb)
mse = mean_squared_error(Y_test, y_pred_gb)
rmse = np.sqrt(mse)
r2 = r2_score(Y_test, y_pred_gb)

gb_results = {
    "Model": "Gradient Boosting Regressor",
    "MAE": mae,
    "MSE": mse,
    "RMSE": rmse,
    "R2 Score": r2
}

print(gb_results)

# 🔥 SAVE METRICS
pd.DataFrame([gb_results]).to_csv("gb_regression_metrics.csv", index=False)

# ================================
# 📉 ACTUAL vs PREDICTED PLOT
# ================================

plt.figure()
plt.scatter(Y_test, y_pred_gb)
plt.xlabel("Actual Lead Score")
plt.ylabel("Predicted Lead Score")
plt.title("Actual vs Predicted - Gradient Boosting")
plt.savefig("actual_vs_pred_gb.png")
plt.close()

# ================================
# 📊 RESIDUAL PLOT
# ================================

residuals = Y_test - y_pred_gb

plt.figure()
plt.scatter(y_pred_gb, residuals)
plt.xlabel("Predicted")
plt.ylabel("Residuals")
plt.title("Residual Plot - Gradient Boosting")
plt.savefig("residual_plot_gb.png")
plt.close()

# ================================
# 📈 FEATURE IMPORTANCE
# ================================

feature_importance = gb_model.feature_importances_

importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": feature_importance
}).sort_values(by="Importance", ascending=False)

importance_df.to_csv("feature_importance_gb.csv", index=False)

# Plot
plt.figure()
plt.barh(importance_df["Feature"], importance_df["Importance"])
plt.title("Feature Importance - Gradient Boosting")
plt.gca().invert_yaxis()
plt.savefig("feature_importance_gb.png")
plt.close()

# ================================
# 💾 SAVE MODEL + SCALER
# ================================

joblib.dump(gb_model, "lead_scoring_model.pkl")
joblib.dump(scaler, "scaler.pkl")

{'Model': 'Gradient Boosting Regressor', 'MAE': 0.4509220555098035, 'MSE': 0.36676569568028855, 'RMSE': 0.6056118358158867, 'R2 Score': 0.998128924487308}


['scaler.pkl']

In [92]:
y_pred_gb[:10]

array([41.82379085, 48.77821539, 31.4619772 , 14.93942054, 25.19953706,
       29.6874683 , 49.87708562, 29.18249045, 22.76048666, 30.35371258])